SỬ DỤNG PYTHON 3.12 TRÊN KAGGLE ĐỂ HUẤN LUYỆN MÔ HÌNH PhoBERT + GRU

In [1]:
# =====================================================================
# CELL 1: CẬP NHẬT VÀ CÀI ĐẶT HỆ SINH THÁI HUGGING FACE (Lặng lẽ)
# =====================================================================
# Dùng tham số -q (quiet) để ẩn đi các dòng log cài đặt dài dòng
!pip install -q transformers datasets evaluate accelerate scikit-learn
print("✅ Cài đặt xong! Hệ sinh thái Hugging Face đã sẵn sàng.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 100.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 2

In [2]:
# =====================================================================
# CELL 2: NẠP THƯ VIỆN & KIỂM TRA PHẦN CỨNG (ZERO TRUST)
# =====================================================================
import os
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer

# Chốt chặn kiểm tra: Ép hệ thống báo cáo đang dùng GPU hay CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Hệ thống tính toán đang chạy trên: [{device.type.upper()}]")

if device.type != 'cuda':
    print("❌ CẢNH BÁO: Chưa bật GPU! Hãy vào menu bên phải -> Accelerator -> Chọn GPU T4 x2.")
else:
    print(f"✅ GPU sẵn sàng: {torch.cuda.get_device_name(0)}")

🚀 Hệ thống tính toán đang chạy trên: [CUDA]
✅ GPU sẵn sàng: Tesla T4


In [3]:
# =====================================================================
# CELL 3: TỰ ĐỘNG TÌM FILE & NẠP TOKENIZER
# =====================================================================
import os
import pandas as pd
from transformers import AutoTokenizer
from datasets import Dataset, DatasetDict

# 1. Zero Trust: Không gõ tay đường dẫn, bắt máy tự tìm file CSV
print("🔍 Đang rà quét toàn bộ kho dữ liệu Kaggle...")
train_path = None
dev_path = None

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename == 'train_clean_PhoBERT.csv':
            train_path = os.path.join(dirname, filename)
        elif filename == 'dev_clean_PhoBERT.csv':
            dev_path = os.path.join(dirname, filename)

if not train_path or not dev_path:
    raise FileNotFoundError("❌ LỖI NGHIÊM TRỌNG: Không tìm thấy file. Bạn đã nhấn 'Add Input' để nạp dataset vào Notebook chưa?")

print(f"✅ Đã chốt tọa độ Train: {train_path}")
print(f"✅ Đã chốt tọa độ Dev: {dev_path}")

# 2. Cấu hình Mục tiêu học
TARGET_LABEL = "sentiment" # Nếu bạn muốn đoán chủ đề, đổi thành "topic"

print(f"\n📥 Đang nạp dữ liệu từ tọa độ đã chốt...")
df_train = pd.read_csv(train_path)
df_dev = pd.read_csv(dev_path)

# 3. Tự động mã hóa Nhãn (String -> Integer)
unique_labels = df_train[TARGET_LABEL].dropna().unique().tolist()
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for label, i in label2id.items()}

print(f"✅ Hệ thống nhận diện {len(unique_labels)} nhãn: {label2id}")

# Ép kiểu dữ liệu
df_train['labels'] = df_train[TARGET_LABEL].map(label2id)
df_dev['labels'] = df_dev[TARGET_LABEL].map(label2id)

# 4. Chuyển đổi sang Dataset Hugging Face
hg_dataset = DatasetDict({
    'train': Dataset.from_pandas(df_train[['clean_text_PhoBERT', 'labels']]),
    'validation': Dataset.from_pandas(df_dev[['clean_text_PhoBERT', 'labels']])
})

# 5. Kích hoạt Tokenizer
print("\n⏳ Đang tải PhoBERT Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")

def tokenize_function(examples):
    return tokenizer(examples["clean_text_PhoBERT"], padding="max_length", truncation=True, max_length=256)

print("⚙️ Đang mã hóa hàng loạt văn bản (Batch Tokenization)...")
tokenized_datasets = hg_dataset.map(tokenize_function, batched=True)

print("🎉 Hoàn tất! Dữ liệu đã sẵn sàng để đưa vào GPU.")

🔍 Đang rà quét toàn bộ kho dữ liệu Kaggle...
✅ Đã chốt tọa độ Train: /kaggle/input/datasets/conbobietbay/phobert-student-feedback-clean/train_clean_PhoBERT.csv
✅ Đã chốt tọa độ Dev: /kaggle/input/datasets/conbobietbay/phobert-student-feedback-clean/dev_clean_PhoBERT.csv

📥 Đang nạp dữ liệu từ tọa độ đã chốt...
✅ Hệ thống nhận diện 3 nhãn: {2: 0, 0: 1, 1: 2}

⏳ Đang tải PhoBERT Tokenizer...


config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

⚙️ Đang mã hóa hàng loạt văn bản (Batch Tokenization)...


Map:   0%|          | 0/11426 [00:00<?, ? examples/s]

Map:   0%|          | 0/1583 [00:00<?, ? examples/s]

🎉 Hoàn tất! Dữ liệu đã sẵn sàng để đưa vào GPU.


In [4]:
# =====================================================================
# CELL 4: KÍCH HOẠT HỆ THỐNG HUẤN LUYỆN LAI (PHOBERT + GRU)
# =====================================================================
import torch
import torch.nn as nn
from transformers import AutoModel, TrainingArguments, Trainer
from transformers.modeling_outputs import SequenceClassifierOutput
import evaluate
import numpy as np

print(f"🧠 Đang nạp lõi PhoBERT + GRU với cấu hình {len(unique_labels)} nhãn phân loại...")

# 1. ĐỊNH NGHĨA KIẾN TRÚC MẠNG LAI (CUSTOM MODEL)
class PhoBERT_GRU_Classifier(nn.Module):
    def __init__(self, num_classes, model_name="vinai/phobert-base-v2", gru_hidden_dim=256):
        super(PhoBERT_GRU_Classifier, self).__init__()
        self.num_classes = num_classes
        
        # Lõi PhoBERT
        self.phobert = AutoModel.from_pretrained(model_name)
        # Giúp Hugging Face Trainer nhận diện cấu hình không bị lỗi
        self.config = self.phobert.config 
        
        # Khối GRU xử lý chuỗi 2 chiều
        self.gru = nn.GRU(
            input_size=self.config.hidden_size, # 768
            hidden_size=gru_hidden_dim, 
            num_layers=1, 
            batch_first=True, 
            bidirectional=True
        )
        
        # Khối phân loại tuyến tính
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(gru_hidden_dim * 2, num_classes) # *2 vì GRU chạy 2 chiều
        )

    def forward(self, input_ids, attention_mask, labels=None):
        # Bước 1: Trích xuất đặc trưng câu từ PhoBERT
        outputs = self.phobert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state # Shape: [batch, seq_len, 768]
        
        # Bước 2: Đưa chuỗi vào GRU
        gru_output, _ = self.gru(sequence_output)
        
        # Bước 3: Lấy kết quả ở mốc thời gian cuối cùng của GRU
        last_step_output = gru_output[:, -1, :]
        
        # Bước 4: Chạy qua lớp Linear để phân loại
        logits = self.classifier(last_step_output)
        
        # Bước 5: Bắt buộc tính Loss nếu đang trong quá trình Train
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_classes), labels.view(-1))
            
        # Trả về format chuẩn của Hugging Face để Trainer hiểu được
        return SequenceClassifierOutput(
            loss=loss,
            logits=logits
        )

# Khởi tạo mô hình
model = PhoBERT_GRU_Classifier(num_classes=len(unique_labels))
# Ép model lên GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 2. CẤU HÌNH ĐO LƯỜNG ĐÁNH GIÁ (METRICS)
metric_acc = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = metric_acc.compute(predictions=predictions, references=labels)
    f1 = metric_f1.compute(predictions=predictions, references=labels, average="macro")
    return {"accuracy": acc["accuracy"], "f1_macro": f1["f1"]}

# 3. CẤU HÌNH THAM SỐ HUẤN LUYỆN (TRAINING ARGUMENTS)
# Lưu ý: Mô hình lai phức tạp hơn, có thể cần batch_size nhỏ hơn (ví dụ 16) nếu Kaggle báo hết VRAM (CUDA Out of Memory)
training_args = TrainingArguments(
    output_dir="./phobert_gru_checkpoints",
    eval_strategy="epoch",          
    save_strategy="epoch",          
    learning_rate=2e-5,             
    per_device_train_batch_size=16, # Đã giảm xuống 16 để tránh tràn RAM
    per_device_eval_batch_size=32,
    num_train_epochs=4,             
    weight_decay=0.01,
    fp16=True,                      
    load_best_model_at_end=True,    
    metric_for_best_model="f1_macro",
    report_to="none"                
)

# 4. KÍCH HOẠT TRAINER
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print("\n🚀 BẮT ĐẦU HUẤN LUYỆN KIẾN TRÚC LAI! HÃY THEO DÕI BẢNG THỐNG KÊ BÊN DƯỚI...\n")
trainer.train()

print("\n💾 Đang lưu trọng số mô hình tốt nhất ra thư mục Output...")
# Vì đây là Custom Model, ta lưu bằng torch.save thay vì trainer.save_model để chắc ăn nhất
torch.save(model.state_dict(), "./best_phobert_gru_model.pth")
tokenizer.save_pretrained("./best_phobert_gru_model_tokenizer")
print("✅ Hệ thống đã hoàn tất toàn bộ tiến trình.")

🧠 Đang nạp lõi PhoBERT + GRU với cấu hình 3 nhãn phân loại...


pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]


🚀 BẮT ĐẦU HUẤN LUYỆN KIẾN TRÚC LAI! HÃY THEO DÕI BẢNG THỐNG KÊ BÊN DƯỚI...



/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,No log,0.189214,0.951358,0.851629
2,0.251329,0.186607,0.951990,0.855901
3,0.137994,0.197446,0.952622,0.862389
4,0.137994,0.205078,0.952622,0.866895


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



💾 Đang lưu trọng số mô hình tốt nhất ra thư mục Output...
✅ Hệ thống đã hoàn tất toàn bộ tiến trình.
